# Algorithm-4 vs Oracle Threshold Calibration
**Reviewer R3.3** — compares detection/relabeling thresholds (TD, TR) chosen by Algorithm 4
(no clean labels — picks by downstream val accuracy) vs an oracle tuned on true noise labels.

| Mode | What it does | Cost |
|------|-------------|------|
| `oracle` | Offline grid scan using ground-truth — instant | zero training |
| `algo4` | Trains one lightweight classifier per (TD, TR) pair | ~N × epochs on GPU |
| `compare` | Prints oracle vs Algorithm-4 degradation | instant (reads CSVs) |

## 1. Colab Setup

In [4]:
!uv sync

Resolved 148 packages in 1ms
Prepared 10 packages in 40.22s                                           
Installed 60 packages in 194ms                              
 + annotated-doc==0.0.4
 + anyio==4.14.0
 + certifi==2026.6.17
 + click==8.4.1
 + contourpy==1.3.3
 + cycler==0.12.1
 + filelock==3.29.4
 + fonttools==4.63.0
 + fsspec==2026.6.0
 + h11==0.16.0
 + hf-xet==1.5.1
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.20.1
 + idna==3.18
 + jinja2==3.1.6
 + joblib==1.5.3
 + kiwisolver==1.5.0
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + matplotlib==3.8.4
 + mdurl==0.1.2
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==1.26.4
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cufft-cu12==11.0.2.54
 + nvidia-curand-cu12==10.3.2.106
 + nvidia-cusolver-cu12==11.4.5.107
 + nvidia-cusparse-cu12==12.1.0.106
 + nvidia-nccl-cu12==2.20.5
 + nvidia-nvjitlink

In [5]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir data
!cp -r drive/MyDrive/FashionMNIST ./data/
!cp -r drive/MyDrive/preds ./

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
mkdir: cannot create directory ‘data’: File exists


In [6]:
!cp -r drive/MyDrive/Colab\ Notebooks/models/ ./
!cp drive/MyDrive/Colab\ Notebooks/cifar-10-python.tar.gz ./data/

## 2. Configuration

In [6]:
# ── Edit these before running ─────────────────────────────────────────────────
DATASET     = 'fashionmnist'   # 'fashionmnist' | 'cifar10' | 'cifar10n'
NOISE_RATIO = '20'             # '20' | '30' | '40'  (or 'n' for cifar10n)
OUTPUT_DIR  = "results/calibration"  # None → results/calibration_<dataset>_<noise_ratio>/

# algo4 training settings
EPOCHS      = 10
PATIENCE    = 4
LIMIT       = 0               # 0 = all pairs; set e.g. 5 for a quick smoke-test
# ─────────────────────────────────────────────────────────────────────────────

## 3. Imports & Helpers

In [1]:
import gc
import json
import math
import os
import pickle
import shutil
import tempfile

import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd

from snd.utils import set_global_seed
from snd.config import FashionMNIST_TRAIN_TRANSFORMS, FashionMNIST_TEST_TRANSFORMS
from snd.cli import get_dataset_config

set_global_seed(42)
print('imports OK')

Files already downloaded and verified
Files already downloaded and verified
imports OK


In [3]:
def load_preds(params):
    dfs = []
    for fold in range(1, params['inner_folds_num'] + 1):
        dfs.append(pd.read_csv(params['prediction_path'].format(fold)))
    return pd.concat(dfs, ignore_index=True)


def build_vote_hist(df, num_classes):
    votes = np.array([[int(x) for x in str(s).split('|')] for s in df['preds']])
    hist = (votes[:, :, None] == np.arange(num_classes)).sum(axis=1)
    return hist, votes.shape[1]


def evaluate_pair(TD, TR, hist, noisy, real, mistakes):
    is_noisy = noisy != real
    flagged  = mistakes >= TD

    meets         = hist >= TR
    has_consensus = meets.any(axis=1)
    relabel       = meets.argmax(axis=1)

    retained = (~flagged) | (flagged & has_consensus)
    relabeled = flagged & has_consensus
    discarded = flagged & ~has_consensus
    cleaned   = np.where(flagged, relabel, noisy)

    resid          = retained & (cleaned != real)
    residual_pct   = 100.0 * resid.sum() / max(int(retained.sum()), 1)
    correct_retained = int((retained & (cleaned == real)).sum())
    clean_yield_pct  = 100.0 * correct_retained / len(noisy)

    tp = int((flagged &  is_noisy).sum())
    fp = int((flagged & ~is_noisy).sum())
    fn = int((~flagged &  is_noisy).sum())
    tn = int((~flagged & ~is_noisy).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy  = (tp + tn) / len(noisy)

    return {
        'td': TD, 'tr': TR,
        'det_precision': round(precision, 4), 'det_recall': round(recall, 4),
        'det_f1': round(f1, 4), 'det_accuracy': round(accuracy, 4),
        'residual_pct': round(residual_pct, 4), 'clean_yield_pct': round(clean_yield_pct, 4),
        'retained': int(retained.sum()), 'relabeled': int(relabeled.sum()),
        'discarded': int(discarded.sum()),
        '_cleaned': cleaned, '_retained_mask': retained,
    }


def make_grid(m):
    """TD in {ceil(0.6m)..m}, TR in {ceil(0.5m)..m} per Algorithm 4 pseudocode."""
    td_lo, tr_lo = math.ceil(0.6 * m), math.ceil(0.5 * m)
    return [(td, tr) for td in range(td_lo, m + 1) for tr in range(tr_lo, m + 1)]


def pick_oracle(rows):
    return max(rows, key=lambda r: r['clean_yield_pct'])


def write_clean_pickle(train_dataset, df_index, retained_mask, cleaned, path, exclude_mask=None):
    """Stream the cleaned (retained) samples to a pickle. If exclude_mask is given, those rows
    are held out (never written) so they can serve as a fixed evaluation set."""
    data = np.asarray(train_dataset.data)
    keep = retained_mask.copy()
    if exclude_mask is not None:
        keep = keep & ~exclude_mask
    with open(path, 'wb') as f:
        for r in np.where(keep)[0]:
            entry = {'data': np.asarray(data[df_index[r]], dtype=np.uint8), 'label': int(cleaned[r])}
            pickle.dump(entry, f)


def eval_on_fixed_set(model, images, labels, transform, device, batch_size=512):
    """Accuracy of a trained model on a FIXED held-out reference set, scored against the labels
    provided (here: the held-out samples' NOISY labels). The same set/labels are reused for every
    (TD, TR), so the noisy-label bias is identical across candidates and cancels in the ranking —
    removing the confound of validating on each cleaned set's own (shrinking, easier) split. It
    still tracks the true objective because predicting a clean sample's noisy label == predicting
    its true label."""
    import torch
    from PIL import Image
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            imgs = images[i:i + batch_size]
            lbls = labels[i:i + batch_size]
            x = torch.stack([transform(Image.fromarray(im)) for im in imgs]).to(device)
            pred = model(x).argmax(1).cpu().numpy()
            correct += int((pred == lbls).sum())
            total   += len(lbls)
    return correct / max(total, 1)


print('helpers OK')

helpers OK


## 4. Load Predictions

In [8]:
import argparse, types

# Build a minimal args namespace so get_dataset_config works without the CLI
_args = types.SimpleNamespace(dataset=DATASET, noise_ratio=NOISE_RATIO)
train_dataset, _, _, _, _, params = get_dataset_config(_args)

out_dir = OUTPUT_DIR or os.path.join('results', f'calibration_{DATASET}_{NOISE_RATIO}')
os.makedirs(out_dir, exist_ok=True)

df        = load_preds(params)
noisy     = df['noisy_label'].to_numpy(int)
real      = df['real_label'].to_numpy(int)
mistakes  = df['mistakes'].to_numpy(int)
df_index  = df['index'].to_numpy(int)
num_classes = int(max(noisy.max(), real.max())) + 1
hist, m   = build_vote_hist(df, num_classes)
grid      = make_grid(m)

# ── Fixed held-out reference set for Algorithm-4 selection ────────────────────
# The SAME samples are held out for every (TD, TR) candidate. They are excluded from
# every candidate's training set and scored against their NOISY labels — no clean labels
# are used. A fixed common yardstick removes the "validate on each cleaned set's own split"
# confound that previously made the signal anti-correlate with test accuracy.
HELDOUT_RATIO = 0.1
_rng = np.random.RandomState(123)
heldout_pos = np.zeros(len(df), dtype=bool)
heldout_pos[_rng.choice(len(df), size=int(HELDOUT_RATIO * len(df)), replace=False)] = True
heldout_images = np.asarray(train_dataset.data)[df_index[heldout_pos]].astype(np.uint8)
heldout_labels = noisy[heldout_pos]

print(f'dataset={DATASET}  noise={NOISE_RATIO}  N={len(df)}  m={m}  '
      f'classes={num_classes}  grid={len(grid)} pairs  out={out_dir}')
print(f'fixed held-out reference: {int(heldout_pos.sum())} samples (scored on noisy labels)')

dataset=fashionmnist  noise=20  N=60000  m=10  classes=10  grid=30 pairs  out=results/calibration
fixed held-out reference: 6000 samples (scored on noisy labels)


## 5. Mode: Oracle (instant — uses ground-truth labels)

In [11]:
rows   = [evaluate_pair(td, tr, hist, noisy, real, mistakes) for (td, tr) in grid]
public = [{k: v for k, v in r.items() if not k.startswith('_')} for r in rows]
pd.DataFrame(public).to_csv(os.path.join(out_dir, 'oracle_grid.csv'), index=False)

o    = pick_oracle(public)
f1td = max(public, key=lambda r: r['det_f1'])['td']

print(f'[oracle] {len(public)} pairs → oracle_grid.csv  (F1-optimal TD={f1td})')
print(f'[oracle] best by clean-yield: TD={o["td"]} TR={o["tr"]}  '
      f'yield={o["clean_yield_pct"]}%  det_F1={o["det_f1"]}  '
      f'residual={o["residual_pct"]}%  retained={o["retained"]}')

oracle_rows = public   # kept for the compare cell

[oracle] 30 pairs → oracle_grid.csv  (F1-optimal TD=9)
[oracle] best by clean-yield: TD=10 TR=5  yield=91.8383%  det_F1=0.8515  residual=7.4832%  retained=59560


## 6. Mode: Algorithm 4 (trains one classifier per grid pair — needs GPU)
Picks (TD, TR) by downstream accuracy on a **single fixed held-out reference set** (the same
held-out samples for every candidate, scored against their **noisy** labels — no clean labels
used). This replaces the original "validate on each cleaned set's own 10% split", whose val set
shrank and got easier as cleaning grew more aggressive (an anti-signal, r≈−0.56 vs test acc).
We log the old signal too (`val_cleaned`) so the run shows both correlations.

In [12]:
import torch
from snd.evaluation.final_model_tester import FinalModelTester

algo4_rows = []
pairs  = grid[:LIMIT] if LIMIT else grid
tmpdir = tempfile.mkdtemp(prefix='algo4_calib_')

try:
    for (td, tr) in pairs:
        m_pair = evaluate_pair(td, tr, hist, noisy, real, mistakes)
        pkl    = os.path.join(tmpdir, f'clean_td{td}_tr{tr}.pkl')
        # exclude the FIXED held-out samples from this candidate's training set
        write_clean_pickle(train_dataset, df_index, m_pair['_retained_mask'], m_pair['_cleaned'],
                           pkl, exclude_mask=heldout_pos)

        tester = FinalModelTester(
            train_dataset_path=pkl,
            train_transform=FashionMNIST_TRAIN_TRANSFORMS,
            test_transform=FashionMNIST_TEST_TRANSFORMS,
            test='fmnist', val_ratio=0.1, patience=PATIENCE,
            smoothing=0.1, cnn_size=512, test_batch_size=512,
            train_batch_size=6000
        )
        tester.train(epochs=EPOCHS)

        # NEW Algorithm-4 signal: accuracy on the FIXED held-out reference set (noisy labels)
        val_fixed   = eval_on_fixed_set(tester.model, heldout_images, heldout_labels,
                                        FashionMNIST_TEST_TRANSFORMS, tester.device)
        # OLD signal (validate on the cleaned set's own split) — kept only for comparison
        val_cleaned = float(tester.best_val_accuracy)
        # true test accuracy on the real FMNIST test set — the oracle reference (not used by Algo-4)
        test_acc    = float(tester.test())

        algo4_rows.append({
            'td': td, 'tr': tr,
            'val_fixed': round(val_fixed, 4), 'val_cleaned': round(val_cleaned, 4),
            'test_accuracy': round(test_acc, 4),
            'det_f1': m_pair['det_f1'], 'residual_pct': m_pair['residual_pct'],
            'clean_yield_pct': m_pair['clean_yield_pct'], 'retained': m_pair['retained'],
            'relabeled': m_pair['relabeled'], 'discarded': m_pair['discarded'],
        })
        print(f'[algo4] TD={td} TR={tr}  val_fixed={val_fixed:.4f}  val_cleaned={val_cleaned:.4f}  '
              f'test_acc={test_acc:.4f}  (retained={m_pair["retained"]})')

        os.remove(pkl)
        del tester
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
finally:
    shutil.rmtree(tmpdir, ignore_errors=True)

pd.DataFrame(algo4_rows).to_csv(os.path.join(out_dir, 'algo4_grid.csv'), index=False)
best_fixed = max(algo4_rows, key=lambda r: r['val_fixed'])
best_test  = max(algo4_rows, key=lambda r: r['test_accuracy'])
print(f'\n[algo4] {len(algo4_rows)} pairs → algo4_grid.csv')
print(f'[algo4] Algorithm-4 pick (max FIXED held-out acc) : TD={best_fixed["td"]} TR={best_fixed["tr"]}  '
      f'val_fixed={best_fixed["val_fixed"]}  test_acc={best_fixed["test_accuracy"]}')
print(f'[algo4] oracle pick       (max true test acc)     : TD={best_test["td"]} TR={best_test["tr"]}  '
      f'test_acc={best_test["test_accuracy"]}')

Loaded 51425 samples from /tmp/algo4_calib_g_rgo1vq/clean_td6_tr5.pkl
Dataset size: 51425


100%|██████████| 10/10 [03:27<00:00, 20.73s/it, Training Loss=0.568, Training Accuracy=0.974, Validation Loss=0.659, Validation Accuracy=0.933]


Test Accuracy: 0.8549
[algo4] TD=6 TR=5  val_fixed=0.6853  val_cleaned=0.9434  test_acc=0.8549  (retained=57103)
Loaded 49747 samples from /tmp/algo4_calib_g_rgo1vq/clean_td6_tr6.pkl
Dataset size: 49747


100%|██████████| 10/10 [03:20<00:00, 20.06s/it, Training Loss=0.547, Training Accuracy=0.985, Validation Loss=0.594, Validation Accuracy=0.96]


Test Accuracy: 0.8582
[algo4] TD=6 TR=6  val_fixed=0.6927  val_cleaned=0.9598  test_acc=0.8582  (retained=55227)
Loaded 47902 samples from /tmp/algo4_calib_g_rgo1vq/clean_td6_tr7.pkl
Dataset size: 47902


...

Test Accuracy: 0.8924
[algo4] TD=10 TR=8  val_fixed=0.7113  val_cleaned=0.9032  test_acc=0.8924  (retained=57472)
Loaded 50969 samples from /tmp/algo4_calib_g_rgo1vq/clean_td10_tr9.pkl
Dataset size: 50969


100%|██████████| 10/10 [03:24<00:00, 20.50s/it, Training Loss=0.673, Training Accuracy=0.934, Validation Loss=0.768, Validation Accuracy=0.892]


Test Accuracy: 0.8939
[algo4] TD=10 TR=9  val_fixed=0.7142  val_cleaned=0.9046  test_acc=0.8939  (retained=56619)
Loaded 49568 samples from /tmp/algo4_calib_g_rgo1vq/clean_td10_tr10.pkl
Dataset size: 49568


100%|██████████| 10/10 [03:21<00:00, 20.14s/it, Training Loss=0.676, Training Accuracy=0.932, Validation Loss=0.79, Validation Accuracy=0.886]


Test Accuracy: 0.8788
[algo4] TD=10 TR=10  val_fixed=0.7062  val_cleaned=0.8890  test_acc=0.8788  (retained=55058)

[algo4] 30 pairs → algo4_grid.csv
[algo4] Algorithm-4 pick (max FIXED held-out acc) : TD=10 TR=6  val_fixed=0.7147  test_acc=0.8912
[algo4] oracle pick       (max true test acc)     : TD=10 TR=9  test_acc=0.8939


## 7. Mode: Compare (instant — reads saved CSVs)

In [9]:
oracle_csv = os.path.join(out_dir, 'oracle_grid.csv')
algo4_csv  = os.path.join(out_dir, 'algo4_grid.csv')

offline = (pd.read_csv(oracle_csv).to_dict('records')
           if os.path.exists(oracle_csv) else oracle_rows)

if not os.path.exists(algo4_csv):
    print('[compare] algo4_grid.csv not found — run the algo4 cell first.')
else:
    a4_rows = pd.read_csv(algo4_csv).to_dict('records')

    algo4  = max(a4_rows, key=lambda r: r['val_fixed'])      # Algorithm-4: fixed held-out, no clean labels
    oracle = max(a4_rows, key=lambda r: r['test_accuracy'])  # oracle: true held-out test acc
    f1_td  = max(offline,  key=lambda r: r['det_f1'])['td']
    keys   = ('td', 'tr', 'test_accuracy', 'det_f1', 'residual_pct', 'clean_yield_pct')

    # Does each label-free signal track the true objective? (the whole point of the fix)
    vf = np.array([r['val_fixed']     for r in a4_rows])
    vc = np.array([r['val_cleaned']   for r in a4_rows])
    ta = np.array([r['test_accuracy'] for r in a4_rows])
    corr_fixed   = float(np.corrcoef(vf, ta)[0, 1])
    corr_cleaned = float(np.corrcoef(vc, ta)[0, 1])

    comparison = {
        'oracle_by_test_acc':        {k: oracle[k] for k in keys},
        'algorithm4_by_val_fixed':   {**{k: algo4[k] for k in keys}, 'val_fixed': algo4['val_fixed']},
        'downstream_test_acc_degradation_pp': round(100 * (oracle['test_accuracy'] - algo4['test_accuracy']), 3),
        'signal_corr_with_test_acc': {'fixed_heldout_NEW': round(corr_fixed, 3),
                                      'cleaned_split_OLD': round(corr_cleaned, 3)},
        'reference': {'offline_F1_optimal_TD': f1_td},
    }
    with open(os.path.join(out_dir, 'comparison.json'), 'w') as f:
        json.dump(comparison, f, indent=2)

    print('\n[compare] oracle (true test acc) vs Algorithm-4 (fixed held-out, no clean labels)')
    print(f'  oracle      : TD={oracle["td"]} TR={oracle["tr"]}  test_acc={oracle["test_accuracy"]}  '
          f'det_F1={oracle["det_f1"]}  residual={oracle["residual_pct"]}%')
    print(f'  Algorithm-4 : TD={algo4["td"]} TR={algo4["tr"]}  test_acc={algo4["test_accuracy"]}  '
          f'(picked by val_fixed={algo4["val_fixed"]})  det_F1={algo4["det_f1"]}  residual={algo4["residual_pct"]}%')
    print(f'  degradation : {comparison["downstream_test_acc_degradation_pp"]} pp')
    print(f'  signal->test-acc correlation:  NEW fixed held-out r={corr_fixed:+.3f}   '
          f'OLD cleaned-split r={corr_cleaned:+.3f}')

    print('\ncomparison.json:')
    print(json.dumps(comparison, indent=2))


[compare] oracle (true test acc) vs Algorithm-4 (fixed held-out, no clean labels)
  oracle      : TD=10 TR=9  test_acc=0.8939  det_F1=0.8515  residual=5.3586%
  Algorithm-4 : TD=10 TR=6  test_acc=0.8912  (picked by val_fixed=0.7147)  det_F1=0.8515  residual=6.8594%
  degradation : 0.27 pp
  signal->test-acc correlation:  NEW fixed held-out r=+0.954   OLD cleaned-split r=-0.307

comparison.json:
{
  "oracle_by_test_acc": {
    "td": 10,
    "tr": 9,
    "test_accuracy": 0.8939,
    "det_f1": 0.8515,
    "residual_pct": 5.3586,
    "clean_yield_pct": 89.3083
  },
  "algorithm4_by_val_fixed": {
    "td": 10,
    "tr": 6,
    "test_accuracy": 0.8912,
    "det_f1": 0.8515,
    "residual_pct": 6.8594,
    "clean_yield_pct": 91.4517,
    "val_fixed": 0.7147
  },
  "downstream_test_acc_degradation_pp": 0.27,
  "signal_corr_with_test_acc": {
    "fixed_heldout_NEW": 0.954,
    "cleaned_split_OLD": -0.307
  },
  "reference": {
    "offline_F1_optimal_TD": 9
  }
}


In [10]:
# ── R3.3 correlation: noise-detection F1 (per TD) vs downstream probe accuracy ───────────────
# `det_f1` is computed from the same ensemble votes as the paper, so it equals the FMNIST-20 report's
# Noise F1 (0.8572 @ TD9, 0.8515 @ TD10). Detection depends only on TD, so we report the correlation
# per-TD (F1 vs the mean/max probe accuracy over TR) and across all 30 (TD, TR) pairs -- the aggregation
# is explicit so a reviewer can reproduce it from algo4_grid.csv with no GPU.
import numpy as np, pandas as pd, os
_df = pd.read_csv(os.path.join(out_dir, 'algo4_grid.csv'))
_per_td = (_df.groupby('td')
             .agg(det_f1=('det_f1', 'first'),
                  mean_test_acc=('test_accuracy', 'mean'),
                  max_test_acc=('test_accuracy', 'max')).reset_index())

def _r(x, y):
    return float(np.corrcoef(np.asarray(x, float), np.asarray(y, float))[0, 1])

print(_per_td.round(4).to_string(index=False))
print("\nnoise-detection F1 vs downstream probe accuracy:")
print(f"  per-TD  F1 vs mean acc (n={len(_per_td)}) : r = {_r(_per_td.det_f1, _per_td.mean_test_acc):+.3f}")
print(f"  per-TD  F1 vs max  acc             : r = {_r(_per_td.det_f1, _per_td.max_test_acc):+.3f}")
print(f"  all {len(_df)} (TD,TR) pairs           : r = {_r(_df.det_f1, _df.test_accuracy):+.3f}")
print(f"  [ref] label-free signal val_fixed vs test acc : r = {_r(_df.val_fixed, _df.test_accuracy):+.3f}")

 td  det_f1  mean_test_acc  max_test_acc
  6  0.7618         0.8668        0.8790
  7  0.7959         0.8704        0.8746
  8  0.8319         0.8722        0.8848
  9  0.8572         0.8771        0.8839
 10  0.8515         0.8830        0.8939

noise-detection F1 vs downstream probe accuracy:
  per-TD  F1 vs mean acc (n=5) : r = +0.868
  per-TD  F1 vs max  acc             : r = +0.716
  all 30 (TD,TR) pairs           : r = +0.509
  [ref] label-free signal val_fixed vs test acc : r = +0.954
